In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test").getOrCreate()

data = [("Alice", 25), ("Bob", 30)]
df = spark.createDataFrame(data, ["Name", "Age"])

df.show()

+-----+---+
| Name|Age|
+-----+---+
|Alice| 25|
|  Bob| 30|
+-----+---+



In [ ]:
from google.colab import files
files.upload()

Saving product_data.csv to product_data.csv
Saving customer_data.csv to customer_data (1).csv
Saving sales_data.csv to sales_data.csv


{'product_data.csv': b'product_id,category,brand\n201,Electronics,Sony\n202,Clothing,Zara\n203,Grocery,Nestle\n204,Electronics,Samsung\n205,Clothing,H&M\n206,Grocery,Amul\n207,Electronics,LG\n208,Clothing,Uniqlo\n209,Grocery,Tata\n210,Electronics,Apple\n',
 'customer_data (1).csv': b'customer_id,name,city,segment\n101,Amit,Delhi,Gold\n102,Neha,Mumbai,Silver\n103,Rahul,Bangalore,Gold\n104,Sneha,Delhi,Bronze\n105,Vikas,Pune,Silver\n106,Riya,Mumbai,Gold\n107,Arjun,Delhi,Bronze\n108,Pooja,Bangalore,Silver\n109,Karan,Pune,Gold\n110,Meera,Delhi,Bronze\n',
 'sales_data.csv': b'transaction_id,customer_id,product_id,store_id,quantity,price,timestamp\n1,101,201,1,2,500,2026-01-01\n2,102,202,1,1,700,2026-01-02\n3,103,203,2,3,300,2026-01-03\n4,104,204,2,2,400,2026-01-04\n5,105,205,3,1,900,2026-01-05\n6,101,201,1,4,500,2026-01-06\n7,102,202,1,2,700,2026-01-07\n8,103,203,2,1,300,2026-01-08\n9,104,204,2,5,400,2026-01-09\n10,105,205,3,2,900,2026-01-10\n11,106,206,1,1,600,2026-01-11\n12,107,207,2,3,800

In [ ]:
rdd = spark.sparkContext.textFile("sales_data.csv")

header = rdd.first()
rdd = rdd.filter(lambda row: row != header)

rdd = rdd.map(lambda row: row.split(","))

In [ ]:
mapped = rdd.map(lambda x: (x[2], int(x[4]) * float(x[5])))

In [ ]:
filtered = mapped.filter(lambda x: x[1] > 1000)

In [ ]:
result = filtered.reduceByKey(lambda a, b: a + b)

result.collect()

[('201', 2000.0),
 ('202', 1400.0),
 ('204', 2000.0),
 ('205', 1800.0),
 ('207', 2400.0),
 ('209', 6000.0),
 ('206', 1200.0)]

In [ ]:
sales_df = spark.read.csv("sales_data.csv", header=True, inferSchema=True)
customer_df = spark.read.csv("customer_data.csv", header=True, inferSchema=True)
product_df = spark.read.csv("product_data.csv", header=True, inferSchema=True)

In [ ]:
full_df = sales_df \
    .join(customer_df, "customer_id") \
    .join(product_df, "product_id")

In [ ]:
from pyspark.sql.functions import col

full_df = full_df.withColumn("revenue", col("quantity") * col("price"))

In [ ]:
from pyspark.sql.functions import sum

full_df.groupBy("city").agg(sum("revenue").alias("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         2200|
|   Mumbai|         3900|
|     Pune|         8700|
|    Delhi|        10500|
+---------+-------------+



In [ ]:
from pyspark.sql.functions import sum

full_df.groupBy("city").agg(sum("revenue").alias("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         2200|
|   Mumbai|         3900|
|     Pune|         8700|
|    Delhi|        10500|
+---------+-------------+



In [ ]:
full_df.groupBy("product_id") \
    .agg(sum("revenue").alias("revenue")) \
    .orderBy("revenue", ascending=False) \
    .show(5)

+----------+-------+
|product_id|revenue|
+----------+-------+
|       209|   6000|
|       207|   3200|
|       201|   3000|
|       204|   2800|
|       205|   2700|
+----------+-------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import month

full_df.groupBy(month("timestamp").alias("month")) \
    .agg(sum("revenue").alias("revenue")) \
    .orderBy("month") \
    .show()

+-----+-------+
|month|revenue|
+-----+-------+
|    1|  25300|
+-----+-------+

